# SQL for Data Extraction: Python + SQLAlchemy

**Task 2 · Day 11-13: Python + SQL Integration**

This notebook connects Python to a SQLite database with SQLAlchemy, runs SQL with `pandas.read_sql()`, and answers 10 business questions.

## 1. Setup and connection

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text, inspect

from database_utils import create_sample_database, get_engine, run_query

DB_PATH = "sales_database.db"

# Build the sample database if it doesn't exist yet
if not os.path.exists(DB_PATH):
    create_sample_database(DB_PATH)

engine = get_engine(DB_PATH)
print("Tables:", inspect(engine).get_table_names())

## 2. Explore the data with `pandas.read_sql()`

In [ ]:
with engine.connect() as conn:
    customers = pd.read_sql(text("SELECT * FROM customers LIMIT 5"), conn)
customers

In [ ]:
for table in ["customers", "products", "orders", "order_items"]:
    n = run_query(engine, f"SELECT COUNT(*) AS n FROM {table}").iloc[0, 0]
    print(f"{table:<12} {n:>5} rows")

## 3. Ten business questions answered with SQL

### Q1. Top 5 products by sales

In [ ]:
q1 = run_query(engine, """
    SELECT p.name AS product, p.category,
           SUM(oi.quantity) AS units_sold,
           ROUND(SUM(oi.quantity * oi.unit_price), 2) AS revenue
    FROM order_items oi
    JOIN products p ON p.product_id = oi.product_id
    JOIN orders   o ON o.order_id   = oi.order_id
    WHERE o.status = 'Completed'
    GROUP BY p.product_id, p.name, p.category
    ORDER BY revenue DESC
    LIMIT 5
""")
q1

In [ ]:
q1.plot(kind="barh", x="product", y="revenue", legend=False, title="Top 5 products by revenue")
plt.gca().invert_yaxis()
plt.xlabel("Revenue")
plt.show()

### Q2. Monthly sales trend

In [ ]:
q2 = run_query(engine, """
    SELECT order_month AS month, revenue FROM vw_monthly_sales ORDER BY order_month
""")
q2.plot(x="month", y="revenue", marker="o", legend=False, title="Monthly revenue")
plt.xticks(rotation=45)
plt.show()
q2.head()

### Q3. Customer segmentation by spend (Gold / Silver / Bronze)

In [ ]:
q3 = run_query(engine, """
    SELECT segment, COUNT(*) AS customers,
           ROUND(SUM(total_spent), 2) AS revenue,
           ROUND(AVG(total_spent), 2) AS avg_spend
    FROM vw_customer_segments
    GROUP BY segment
    ORDER BY revenue DESC
""")
q3

### Q4. Revenue by city

In [ ]:
run_query(engine, """
    SELECT city, COUNT(DISTINCT customer_id) AS customers,
           COUNT(DISTINCT order_id) AS orders,
           ROUND(SUM(line_total), 2) AS revenue
    FROM vw_sales_detail
    GROUP BY city
    ORDER BY revenue DESC
""")

### Q5. Revenue by product category

In [ ]:
run_query(engine, """
    SELECT category, SUM(quantity) AS units_sold, ROUND(SUM(line_total), 2) AS revenue
    FROM vw_sales_detail
    GROUP BY category
    ORDER BY revenue DESC
""")

### Q6. Top 5 customers by lifetime spend

In [ ]:
run_query(engine, """
    SELECT customer_name, city, orders, total_spent
    FROM vw_customer_segments
    ORDER BY total_spent DESC
    LIMIT 5
""")

### Q7. Average order value (AOV) by month

In [ ]:
run_query(engine, """
    SELECT order_month,
           ROUND(SUM(line_total) / COUNT(DISTINCT order_id), 2) AS avg_order_value
    FROM vw_sales_detail
    GROUP BY order_month
    ORDER BY order_month
""")

### Q8. Repeat customers: who has ordered more than 5 times?

In [ ]:
run_query(engine, """
    SELECT c.name, c.city, COUNT(*) AS completed_orders
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
    WHERE o.status = 'Completed'
    GROUP BY c.customer_id, c.name, c.city
    HAVING COUNT(*) > 5
    ORDER BY completed_orders DESC
""")

### Q9. Best-selling product in each category (window function)

In [ ]:
run_query(engine, """
    WITH ranked AS (
        SELECT category, product_name,
               SUM(line_total) AS revenue,
               RANK() OVER (PARTITION BY category ORDER BY SUM(line_total) DESC) AS rnk
        FROM vw_sales_detail
        GROUP BY category, product_name
    )
    SELECT category, product_name, ROUND(revenue, 2) AS revenue
    FROM ranked
    WHERE rnk = 1
""")

### Q10. Cancellation and return rate

In [ ]:
run_query(engine, """
    SELECT status, COUNT(*) AS orders,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM orders), 1) AS pct_of_orders
    FROM orders
    GROUP BY status
    ORDER BY orders DESC
""")

## 4. Reusable helper
All the connection and query logic lives in `database_utils.py`, so any other script or notebook can reuse it:

```python
from database_utils import get_engine, run_query
df = run_query(get_engine("sales_database.db"), "SELECT * FROM products")
```